# Lesson 04 - 도구 사용 디자인 패턴

이 강의에서는 Microsoft Agent Framework (Python)를 사용한 AI 에이전트를 위한 **도구 사용** 디자인 패턴을 배웁니다. 다루는 내용은 다음과 같습니다:

- `@tool` 데코레이터와 타입이 지정된 매개변수를 사용하여 함수 도구 정의
- 각 도구가 하는 일을 모델이 알 수 있도록 도구 스키마 제공
- `approval_mode`로 도구 실행 제어
- Pydantic 모델과 `response_format`을 통해 **구조화된 출력** 반환

시나리오는 목적지를 조회하고, 이용 가능 여부를 확인하며, 항공편 정보를 검색할 수 있는 <strong>여행 예약 에이전트</strong>입니다.


## 설정


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Defining Tools with the @tool Decorator

The `@tool` decorator turns a plain Python function into a tool that an agent can call.
Key points:

- The **docstring** becomes the tool description the model sees.
- **Type annotations** (including `Annotated` with descriptions) define the tool schema.
- `approval_mode` controls whether the user must approve each call before it executes.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## 여러 도구를 사용하는 에이전트 만들기

세 가지 도구를 모두 클라이언트에 전달하여 모델이 사용자 질문에 답하기 위해 필요한 도구를 호출할 수 있도록 합니다.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## 도구를 사용한 구조화된 출력

`response_format`을 Pydantic 모델로 설정하면 에이전트가 자유 형식 텍스트 대신 잘 타입된 JSON 객체를 반환하도록 강제합니다. 이는 하위 코드가 결과를 프로그래밍 방식으로 처리해야 할 때 유용합니다.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## 도구 승인 패턴

`@tool`의 `approval_mode` 매개변수는 도구 호출이 실행되기 전에 인간의 승인이 필요한지 여부를 제어합니다:

| 모드 | 동작 |
|---|---|
| `"never_require"` | 도구가 자동으로 실행됩니다 — 사용자 확인이 필요 없습니다. |
| `"always_require"` | 모든 호출은 실행 전에 사용자의 승인을 받아야 합니다. |

부작용이 있는 도구(예: 항공편 예약, 신용카드 결제)에는 `"always_require"`를 사용하여 사람이 개입하도록 하세요.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## 요약

이 수업에서 배운 내용:

1. 타입이 지정된 매개변수와 도구 스키마 역할을 하는 docstring을 사용하여 `@tool` 데코레이터로 <strong>도구 정의</strong>하기.
2. 에이전트가 복잡한 쿼리를 처리할 수 있도록 여러 도구를 <strong>연속적으로 호출하도록 구성</strong>하기.
3. `response_format`으로 Pydantic 모델을 전달하여 <strong>구조화된 출력 반환</strong>하기.
4. 민감한 작업에 대해 사람의 승인을 유지하기 위해 `approval_mode`로 <strong>도구 승인 제어</strong>하기.

이러한 패턴은 외부 시스템과 안전하게 상호작용할 수 있는 신뢰성 있고 프로덕션 준비가 된 에이전트를 구축하는 토대가 됩니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
